# Composition Study — Data Explorer & Visualizer

Demonstrates `data_parser.py` and `visualizer.py` on the collected experiment data.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

import data_parser as dp
import visualizer as viz

---
## 1. Load Data

In [ ]:
data = dp.load_all_data(only_completed=True)

participants = data['participants']
annotations  = data['annotations']
shapes       = data['shapes']
ratings      = data['ratings']

print(f"Completed participants : {len(participants)}")
print(f"Annotation trials      : {len(annotations)}")
print(f"Individual shapes drawn: {len(shapes)}")
print(f"Aesthetic ratings      : {len(ratings)}")

---
## 2. Participants Overview

In [ ]:
participants[[
    'participant_id', 'image_type', 'combination',
    'age', 'gender', 'visual_expertise', 'EMS',
    'total_time_sec', 'num_annotations', 'num_ratings'
]]

---
## 3. Shape Distribution

In [ ]:
# Count shape figures by condition
fig_counts = shapes.groupby(['condition', 'figure']).size().unstack(fill_value=0)
fig_counts.plot(kind='bar', figsize=(12, 4), title='Shape Figures by Condition')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("\nShape figure totals:")
print(shapes['figure'].value_counts())

---
## 4. Aesthetic Ratings

In [ ]:
# Top and bottom rated images
rating_summary = (
    ratings.groupby('image_filename')['rating']
    .agg(['mean', 'std', 'count'])
    .sort_values('mean', ascending=False)
)

print("Top 5 highest-rated images:")
print(rating_summary.head())
print("\nBottom 5 lowest-rated images:")
print(rating_summary.tail())

# Distribution plot
ratings['rating'].plot(kind='hist', bins=7, figsize=(7,3), title='Aesthetic Rating Distribution', edgecolor='black')
plt.xlabel('Rating (1–7)')
plt.tight_layout()
plt.show()

---
## 5. Single-Image Annotation Summary

`viz.plot_annotation_summary()` renders:
- **Original image**
- **All annotated shapes** (filled polygons / contour lines)
- **Density heatmap** (Gaussian-smoothed coverage heat)

Optionally filter to a specific `condition`.

In [ ]:
# Pick the most-annotated image across all sessions
top_img = shapes['image_filename'].value_counts().idxmax()
n_shapes = shapes['image_filename'].value_counts().max()
print(f"Most annotated image: {top_img}  ({n_shapes} shapes across all sessions/conditions)")

fig = viz.plot_annotation_summary(
    image_filename=top_img,
    df_shapes=shapes,
    base_dir='images',
    condition=None,   # None = all conditions combined
    figsize=(16, 5),
)
plt.show()

---
## 6. Condition-Filtered Annotation View

Show only the shapes from **Category A** (e.g. `Emphasis`) or **Category B** (e.g. `Layering`) for a given image.

In [ ]:
# Find available conditions in the data
available_conditions = shapes['condition'].dropna().unique().tolist()
print("Available conditions:", available_conditions)

# Pick the first condition
cond = available_conditions[0]
cond_img = shapes[shapes['condition'] == cond]['image_filename'].value_counts().idxmax()
print(f"\nShowing condition '{cond}' on image: {cond_img}")

fig = viz.plot_annotation_summary(
    image_filename=cond_img,
    df_shapes=shapes,
    base_dir='images',
    condition=cond,
    figsize=(16, 5),
)
plt.show()

---
## 7. Side-by-Side Condition Comparison

`viz.compare_conditions_heatmap()` compares the annotation density for the **same image** across two different compositional conditions — directly equivalent to the `trial_type` category pairings (e.g. the `Balance` vs `Segregating` aspects of a `BS` trial).

In [ ]:
# Find an image that appears under at least 2 different conditions
img_cond_counts = shapes.groupby('image_filename')['condition'].nunique()
multi_cond_imgs = img_cond_counts[img_cond_counts >= 2].index.tolist()

if not multi_cond_imgs:
    print("No image appears under 2+ conditions yet (need more participants).")
else:
    chosen_img = multi_cond_imgs[0]
    conds = shapes[shapes['image_filename'] == chosen_img]['condition'].unique().tolist()
    cond_a, cond_b = conds[0], conds[1]
    print(f"Comparing '{cond_a}' vs '{cond_b}' on: {chosen_img}")

    fig = viz.compare_conditions_heatmap(
        image_filename=chosen_img,
        df_shapes=shapes,
        condition_a=cond_a,
        condition_b=cond_b,
        base_dir='images',
        blur=15,
        figsize=(16, 5),
    )
    plt.show()

---
## 8. Manual Drawing Call

You can also call the primitives directly for custom plots.
Here we use the low-level `get_total_mask` + `create_heatmap` to render only `custom_contour` shapes (the freehand lines).

In [ ]:
import cv2  # noqa

# Pick image and filter to contour shapes only
img_name = shapes['image_filename'].value_counts().idxmax()
df_contours = shapes[
    (shapes['image_filename'] == img_name) &
    (shapes['figure'] == 'custom_contour')
]

img = viz.load_image(img_name, base_dir='images')

# Labelled (outlines only, no fill)
labelled_no_fill = viz.label_image(img, df_contours, fill=False, line_thickness=3)

# Density mask and heatmap
mask = viz.get_total_mask(img, df_contours, line_thickness=8)
heatmap = viz.create_heatmap(img, mask, blur=20, colormap=cv2.COLORMAP_TURBO, alpha=0.65)

fig, axs = plt.subplots(1, 3, figsize=(16, 5))
axs[0].imshow(img)
axs[0].set_title('Original')
axs[0].axis('off')

axs[1].imshow(labelled_no_fill)
axs[1].set_title(f'Contour Lines Only (N={len(df_contours)})')
axs[1].axis('off')

axs[2].imshow(heatmap)
axs[2].set_title('Density Heatmap (Turbo cmap)')
axs[2].axis('off')

plt.suptitle(f'{img_name} — custom_contour shapes', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Re-sync trials.js (optional)

After collecting new data, run this to update `trials.js` so `choose_trial.js` balances future participants.

In [2]:
# Uncomment to sync:
dp.sync_trials_js()
print('trials.js updated.')

[INFO] Found 417 files in collected_data/data
[INFO] Parsed successfully: 417 sessions (302 completed), 12080 annotation trials, 28067 shapes, 6040 aesthetic ratings.
[INFO] Updated trials.js with 302 completed participant sessions.


trials.js updated.
